<div align="center">

<img src="Phase%20One/assets/branding/srm-ap-logo.png" width="150" alt="SRM University-AP">

<img src="Phase%20One/assets/branding/qiskit-fall-fest-2026-badge-crop.png" width="80" alt="Qiskit Fall Fest 2026">

# Silver-Qx-Finance

### QFF 2026 · Phase 2 · PS-O2

**Constraint-Aware Portfolio Selection**  
Team Coughing Baby · SRM University-AP × VIT-AP

</div>

---

This notebook is the reproducible end-to-end PS-O2 run. It validates the locked inputs, displays the exact classical reference, runs the Phase 2 test suite, then generates the official Processor A/B/D results, D1 Bell verification and the full Processor D penalty sweep.

Run from the repository root. The organizer's `qff_phase2.py` and `data/instances.json` are immutable inputs.


In [1]:
from pathlib import Path
import json, sys, subprocess, csv, runpy, html
from IPython.display import display, Markdown, HTML

ROOT = Path.cwd()
assert (ROOT / "qff_phase2.py").is_file(), "Open this notebook from the Silver-Qx-Finance repository root."
assert (ROOT / "data/instances.json").is_file()
print("Repository:", ROOT)
print("Python:", sys.version.split()[0])
print("D1 selection:", json.loads((ROOT / "processors/processor_D.json").read_text())["selection_id"])


Repository: /home/ashwathraj/Acadimics/3.Semister3/Hackthons/Certificate/QFF IBM  singularity/Silver-Qx-Finance
Python: 3.12.14
D1 selection: D1_K4_TAIL_6Q


## 1. Source integrity and official Processor D validation

The next cell checks the organizer source hashes, the derived PS-O2 instance, D1's topology, basis gates, noise floor, connectivity, qubit count and maximum degree. It stops on a mismatch.


In [2]:
validation = runpy.run_path("scripts/validate_final.py", run_name="validate_final_notebook")
validation["main"]()


PASS locked SHA-256: qff_phase2.py
PASS locked SHA-256: data/instances.json


PASS derived PS-O2 instance matches locked organizer data
PASS official validate_processor_d: D1 K4-tail is valid for PS-O2
PASS D1 topology, gate set, noise floor, and selection marker


0

## 2. Locked formulation and exact classical baseline

The objective is \(F(x)=\mu^T x-qx^T\Sigma x\), constrained to exactly two selected assets. The QUBO minimizes \(-F(x)+\lambda(\sum_i x_i-2)^2\). Every one of the 16 bitstrings is enumerated; six are feasible.


In [3]:
from src.classical_reference import load_instance, enumerate_reference
instance = load_instance(ROOT / "data/instance.json")
reference = enumerate_reference(instance)
print("F* =", reference["classical_optimum_F_star"])
print("Optimal portfolios:", [r["bitstring_x0_to_xn"] for r in reference["optimal_portfolios"]])
print("Uniform-random baseline:", reference["uniform_random_baseline"])

valid_rows = reference["all_valid_portfolios"]
header = ["Portfolio (x0…x3)", "Assets", "F(x)"]
body = "".join(
    "<tr><td><code>{}</code></td><td>{}</td><td>{:.4f}</td></tr>".format(
        r["bitstring_x0_to_xn"], ", ".join(map(str, r["selected_indices"])), r["F"]
    ) for r in valid_rows
)
display(HTML(
    "<table style='border-collapse:collapse;width:100%'>"
    "<thead><tr><th>Portfolio (x0…x3)</th><th>Asset indices</th><th>F(x)</th></tr></thead>"
    "<tbody>" + body + "</tbody></table>"
))


F* = 4.75
Optimal portfolios: ['1100']
Uniform-random baseline: {'number_of_states': 16, 'number_of_valid_states': 6, 'valid_frequency': 0.375, 'p_opt': 0.0625, 'mean_F_over_all_bitstrings': 3.1875, 'mean_F_over_valid_bitstrings': 3.625, 'mean_valid_objective_ratio': 0.7631578947368421}


Portfolio (x0…x3),Asset indices,F(x)
1100,"0, 1",4.7500
0110,"1, 2",4.0000
1001,"0, 3",4.0000
0101,"1, 3",3.5000
0011,"2, 3",3.2500
1010,"0, 2",2.2500


## 3. Official settings

- QAOA \(p=1\); five independent starts (seeds 0–4), \(\gamma,eta\sim U[0,\pi]\).
- COBYLA, maximum 200 exact statevector objective evaluations per start.
- At reference \(\lambda=2\), selected ideal angles are reused unchanged on A/B/D.
- For each required D penalty value \([0,0.5,1,2,5,10]\), the five starts are re-optimized and the selected angles are used for both ideal and noisy sampling.
- 4,096 shots per condition, official Qiskit transpiler settings and synthetic noise model from unchanged `qff_phase2.py`.


## 4. Tests

The root pytest configuration restricts test discovery to the active `tests/` tree, preserving legacy Phase One files without collecting them as Phase 2 tests.


In [4]:
test_run = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(test_run.stdout)
if test_run.returncode:
    raise RuntimeError(f"pytest failed with exit code {test_run.returncode}")


..........                                                               [100%]
10 passed in 1.60s



## 5. Run the full PS-O2 benchmark

This step writes `results/tables/results.csv`, `bell.csv`, processor resource and condition tables, all six D penalty points, optimization-trial metadata, plots, reproducibility metadata, and the generated results sections of `README.md` and `report.md`. It does not modify organizer source files.


In [5]:
runner = runpy.run_path("scripts/run_final_ps_o2.py", run_name="qff_final_runner")
runner["main"]()


PASS official source hash: qff_phase2.py
PASS official source hash: data/instances.json


Optimization λ=0: selected seed=3 E=-4.51246176 gamma=0.78896772 beta=2.66436469


Optimization λ=0.5: selected seed=0 E=-3.30830129 gamma=3.14159265 beta=2.33504824


Optimization λ=1: selected seed=4 E=-3.13244700 gamma=3.06080986 beta=2.42057297


Optimization λ=2: selected seed=0 E=-2.50874217 gamma=3.10823602 beta=0.69980543


Optimization λ=5: selected seed=1 E=-1.37355099 gamma=1.90733813 beta=2.65411574


Optimization λ=10: selected seed=3 E=0.00511187 gamma=0.25190029 beta=1.29514510
Reference λ=2 angles: gamma=3.108236020048, beta=0.699805426888
Processor A ideal: valid=0.467285 P_opt=0.273926 selected=1100 F=4.75 CX=18 depth=42
Processor A noisy: valid=0.448730 P_opt=0.243896 selected=1100 F=4.75 CX=18 depth=42
Processor B ideal: valid=0.473877 P_opt=0.273438 selected=1100 F=4.75 CX=15 depth=34
Processor B noisy: valid=0.455566 P_opt=0.246094 selected=1100 F=4.75 CX=15 depth=34
Processor D ideal: valid=0.467529 P_opt=0.259033 selected=1100 F=4.75 CX=12 depth=23
Processor D noisy: valid=0.459229 P_opt=0.240723 selected=1100 F=4.75 CX=12 depth=23
Bell adjacent pair=(0, 1) ideal: F=1.000000 +/- 0.000000


Bell adjacent pair=(0, 1) noisy: F=0.963257 +/- 0.002092
Bell farthest pair=(1, 5) ideal: F=1.000000 +/- 0.000000
Bell farthest pair=(1, 5) noisy: F=0.949097 +/- 0.002450
D λ=0 ideal: QUBO-min-valid=False valid=0.233643 P_opt=0.089355 ratio=1.0
D λ=0 noisy: QUBO-min-valid=False valid=0.257568 P_opt=0.087402 ratio=1.0
D λ=0.5 ideal: QUBO-min-valid=False valid=0.468994 P_opt=0.119873 ratio=0.6842105263157895
D λ=0.5 noisy: QUBO-min-valid=False valid=0.466797 P_opt=0.116943 ratio=0.6842105263157895
D λ=1 ideal: QUBO-min-valid=True valid=0.454346 P_opt=0.251953 ratio=1.0


D λ=1 noisy: QUBO-min-valid=True valid=0.449951 P_opt=0.236816 ratio=1.0
D λ=2 ideal: QUBO-min-valid=True valid=0.467529 P_opt=0.259033 ratio=1.0
D λ=2 noisy: QUBO-min-valid=True valid=0.459229 P_opt=0.240723 ratio=1.0
D λ=5 ideal: QUBO-min-valid=True valid=0.645264 P_opt=0.163086 ratio=0.47368421052631576
D λ=5 noisy: QUBO-min-valid=True valid=0.619629 P_opt=0.156982 ratio=0.47368421052631576
D λ=10 ideal: QUBO-min-valid=True valid=0.700684 P_opt=0.117676 ratio=0.7368421052631579
D λ=10 noisy: QUBO-min-valid=True valid=0.672363 P_opt=0.115479 ratio=0.7368421052631579



========== FINAL PS-O2 BENCHMARK COMPLETE ==========
Rows in official results.csv: 16
Bell rows: 4
Processor D: D1 K4 tail; selected by three-candidate benchmark
Report: /home/ashwathraj/Acadimics/3.Semister3/Hackthons/Certificate/QFF IBM  singularity/Silver-Qx-Finance/report.md
Results: /home/ashwathraj/Acadimics/3.Semister3/Hackthons/Certificate/QFF IBM  singularity/Silver-Qx-Finance/results/tables/results.csv
PASS: official source/data/Processor A/B files remain byte-for-byte unchanged


0

## 6. Final artifacts

The results below are loaded from the CSV files written by the runner, rather than copied manually into the notebook.


In [6]:
def csv_table(path, max_rows=20):
    with Path(path).open(newline="", encoding="utf-8") as stream:
        rows = list(csv.DictReader(stream))
    if not rows:
        print(f"{path}: no rows")
        return
    headers = list(rows[0].keys())
    shown = rows[:max_rows]
    head = "".join(f"<th>{html.escape(h)}</th>" for h in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{html.escape(str(row.get(h, '')))}</td>" for h in headers) + "</tr>"
        for row in shown
    )
    display(HTML(
        "<div style='overflow-x:auto'><table style='border-collapse:collapse;font-size:12px'>"
        f"<thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>"
    ))
    print(f"{path}: {len(rows)} rows" + (f" (showing {len(shown)})" if len(rows) > max_rows else ""))

csv_table("results/tables/results.csv")
csv_table("results/tables/bell.csv")
csv_table("results/tables/processor_d_lambda_sweep.csv")
print("\nFinal report: report.md")
print("README: README.md")
print("Metadata: results/run_metadata_final.json")


problem_id,processor,condition,executable,logical_qubits,physical_qubits,logical_cx,cx,depth,two_q_depth,swaps,cx_overhead,shots,quality_metric,quality_value,p_opt,classical_optimum,random_baseline,lambda
PS-O2,Processor A,ideal,true,4,5,12,18,42,18,2,6,4096,objective_ratio_selected_valid,1.0,0.27392578125,4.75,0.7631578947368421,2.0
PS-O2,Processor A,noisy,true,4,5,12,18,42,18,2,6,4096,objective_ratio_selected_valid,1.0,0.243896484375,4.75,0.7631578947368421,2.0
PS-O2,Processor B,ideal,true,4,9,12,15,34,15,1,3,4096,objective_ratio_selected_valid,1.0,0.2734375,4.75,0.7631578947368421,2.0
PS-O2,Processor B,noisy,true,4,9,12,15,34,15,1,3,4096,objective_ratio_selected_valid,1.0,0.24609375,4.75,0.7631578947368421,2.0
PS-O2,Processor D,ideal,true,4,6,12,12,23,10,0,0,4096,objective_ratio_selected_valid,1.0,0.259033203125,4.75,0.7631578947368421,2.0
PS-O2,Processor D,noisy,true,4,6,12,12,23,10,0,0,4096,objective_ratio_selected_valid,1.0,0.24072265625,4.75,0.7631578947368421,2.0
PS-O2,Processor D,ideal,true,4,6,12,12,23,10,0,0,4096,objective_ratio_selected_valid,1.0,0.08935546875,4.75,0.7631578947368421,0.0
PS-O2,Processor D,noisy,true,4,6,12,12,23,10,0,0,4096,objective_ratio_selected_valid,1.0,0.08740234375,4.75,0.7631578947368421,0.0
PS-O2,Processor D,ideal,true,4,6,10,10,20,8,0,0,4096,objective_ratio_selected_valid,0.6842105263157895,0.119873046875,4.75,0.7631578947368421,0.5
PS-O2,Processor D,noisy,true,4,6,10,10,20,8,0,0,4096,objective_ratio_selected_valid,0.6842105263157895,0.116943359375,4.75,0.7631578947368421,0.5


results/tables/results.csv: 16 rows


problem_id,processor,pair_type,physical_pair,pair_distance,condition,shots,cx,XX,XX_se,YY,YY_se,ZZ,ZZ_se,fidelity,fidelity_se
PS-O2,Processor D,adjacent,"[0, 1]",1,ideal,4096,1,1.0,0.0,-1.0,0.0,1.0,0.0,1.0,0.0
PS-O2,Processor D,adjacent,"[0, 1]",1,noisy,4096,1,0.94580078125,0.005074178326602111,-0.9521484375,0.004775562205284335,0.955078125,0.004630533407219104,0.9632568359375,0.0020915771516679545
PS-O2,Processor D,farthest,"[1, 5]",3,ideal,4096,7,1.0,0.0,-1.0,0.0,1.0,0.0,1.0,0.0
PS-O2,Processor D,farthest,"[1, 5]",3,noisy,4096,7,0.93408203125,0.005579038766540612,-0.921875,0.006054490609197874,0.9404296875,0.0053123162392081456,0.9490966796875,0.0024495310000179475


results/tables/bell.csv: 4 rows


problem_id,processor,lambda,condition,shots,optimizer,selected_seed,gamma,beta,expected_qubo_energy,qubo_minimum_energy,qubo_minimum_bitstrings,qubo_minimum_valid,valid_solution_frequency,p_opt,selected_portfolio_x0_to_x3,selected_F,best_sampled_valid_portfolio_x0_to_x3,mean_F_valid_samples,objective_ratio_selected_valid,feasible_objective_yield,classical_optimum_F_star,random_baseline_valid_frequency,random_baseline_p_opt,random_baseline_objective_ratio
PS-O2,Processor D,0.0,ideal,4096,COBYLA,3,0.7889677185963135,2.664364685403688,-4.5124617578222725,-5.5,1101,false,0.233642578125,0.08935546875,1100,4.75,1100,4.089602925809823,1.0,0.20115902549342105,4.75,0.375,0.0625,0.7631578947368421
PS-O2,Processor D,0.0,noisy,4096,COBYLA,3,0.7889677185963135,2.664364685403688,-4.5124617578222725,-5.5,1101,false,0.257568359375,0.08740234375,1100,4.75,1100,4.035308056872038,1.0,0.21881424753289475,4.75,0.375,0.0625,0.7631578947368421
PS-O2,Processor D,0.5,ideal,4096,COBYLA,0,3.141592653589793,2.3350482410127014,-3.3083012893570043,-5.0,1101,false,0.468994140625,0.119873046875,0011,3.25,1100,3.739718896408121,0.6842105263157895,0.3692434210526316,4.75,0.375,0.0625,0.7631578947368421
PS-O2,Processor D,0.5,noisy,4096,COBYLA,0,3.141592653589793,2.3350482410127014,-3.3083012893570043,-5.0,1101,false,0.466796875,0.116943359375,0011,3.25,1100,3.73130230125523,0.6842105263157895,0.3666863692434211,4.75,0.375,0.0625,0.7631578947368421
PS-O2,Processor D,1.0,ideal,4096,COBYLA,4,3.0608098606373573,2.420572966607127,-3.132446995038827,-4.75,1100,true,0.454345703125,0.251953125,1100,4.75,1100,4.268538420204191,1.0,0.40829307154605265,4.75,0.375,0.0625,0.7631578947368421
PS-O2,Processor D,1.0,noisy,4096,COBYLA,4,3.0608098606373573,2.420572966607127,-3.132446995038827,-4.75,1100,true,0.449951171875,0.23681640625,1100,4.75,1100,4.220971242539338,1.0,0.39983809621710525,4.75,0.375,0.0625,0.7631578947368421
PS-O2,Processor D,2.0,ideal,4096,COBYLA,0,3.108236020048372,0.6998054268875787,-2.508742173348865,-4.75,1100,true,0.467529296875,0.259033203125,1100,4.75,1100,4.253263707571802,1.0,0.41863692434210525,4.75,0.375,0.0625,0.7631578947368421
PS-O2,Processor D,2.0,noisy,4096,COBYLA,0,3.108236020048372,0.6998054268875787,-2.508742173348865,-4.75,1100,true,0.459228515625,0.24072265625,1100,4.75,1100,4.203216374269005,1.0,0.4063656455592105,4.75,0.375,0.0625,0.7631578947368421
PS-O2,Processor D,5.0,ideal,4096,COBYLA,1,1.9073381273722367,2.6541157405124127,-1.3735509861094637,-4.75,1100,true,0.645263671875,0.1630859375,1010,2.25,1100,3.383654937570942,0.47368421052631576,0.45965254934210525,4.75,0.375,0.0625,0.7631578947368421
PS-O2,Processor D,5.0,noisy,4096,COBYLA,1,1.9073381273722367,2.6541157405124127,-1.3735509861094637,-4.75,1100,true,0.61962890625,0.156982421875,1010,2.25,1100,3.40730890464933,0.47368421052631576,0.4444772820723684,4.75,0.375,0.0625,0.7631578947368421


results/tables/processor_d_lambda_sweep.csv: 12 rows

Final report: report.md
README: README.md
Metadata: results/run_metadata_final.json
